# 1 · Primer contacto con LangGraph

**Objetivo:** entender el modelo mental de LangGraph (**estado, nodos y aristas**) sin LLM de por medio.

Esta sección no necesita clave de API: todo es Python normal.

| Concepto | Qué es | En código |
|---|---|---|
| **Estado** | Los datos compartidos que viajan por el grafo | `TypedDict`, dataclass o Pydantic |
| **Nodo** | Una función que recibe el estado y devuelve **solo lo que cambia** | `def nodo(state) -> dict` |
| **Arista** | Qué nodo va después | `add_edge` (fija) o `add_conditional_edges` (decide una función) |
| **Reducer** | Cómo se combina una actualización con el valor anterior | `Annotated[list, operator.add]` |

El caso: **triaje de incidencias** en Cuac & Co., la empresa ficticia del taller.

## 1.1 · El estado

El estado es un diccionario tipado. `pasos` lleva un **reducer** (`operator.add`): cuando un nodo
devuelve `{"pasos": [...]}`, la lista **se concatena** en lugar de sobrescribirse.

In [ ]:
import operator
from typing import Annotated, Literal, TypedDict

from langgraph.graph import END, START, StateGraph

from taller.viz import mostrar_grafo


class Incidencia(TypedDict):
    texto: str
    severidad: str
    equipo: str
    pasos: Annotated[list[str], operator.add]  # reducer: acumula en vez de sobrescribir

## 1.2 · Los nodos

Un nodo es una función normal. Recibe el estado completo y devuelve una **actualización parcial**.

In [ ]:
def clasificar(state: Incidencia) -> dict:
    texto = state["texto"].lower()
    equipo = "Pagos" if ("pago" in texto or "err-4012" in texto) else "Plataforma"
    severidad = "SEV1" if ("caída" in texto or "no se puede comprar" in texto) else "SEV3"
    return {
        "equipo": equipo,
        "severidad": severidad,
        "pasos": [f"clasificada como {severidad} para {equipo}"],
    }


def avisar_guardia(state: Incidencia) -> dict:
    return {"pasos": [f"aviso enviado a la guardia de {state['equipo']}"]}


def crear_ticket(state: Incidencia) -> dict:
    return {"pasos": [f"ticket creado en la cola de {state['equipo']}"]}

## 1.3 · Las aristas y el enrutado

Una **arista condicional** llama a una función que devuelve el nombre del siguiente nodo.
Anotar el retorno con `Literal[...]` permite a LangGraph dibujar el grafo correctamente.

In [ ]:
def enrutar(state: Incidencia) -> Literal["avisar_guardia", "crear_ticket"]:
    return "avisar_guardia" if state["severidad"] == "SEV1" else "crear_ticket"


builder = StateGraph(Incidencia)
builder.add_node("clasificar", clasificar)
builder.add_node("avisar_guardia", avisar_guardia)
builder.add_node("crear_ticket", crear_ticket)

builder.add_edge(START, "clasificar")
builder.add_conditional_edges("clasificar", enrutar)
builder.add_edge("avisar_guardia", "crear_ticket")
builder.add_edge("crear_ticket", END)

triaje = builder.compile()  # compile() valida el grafo y devuelve algo ejecutable
mostrar_grafo(triaje)

## 1.4 · Ejecutar

`invoke` recibe el estado inicial y devuelve el estado final.

In [ ]:
triaje.invoke({"texto": "La web está caída y no se puede comprar con ERR-4012"})

In [ ]:
triaje.invoke({"texto": "El logo sale pixelado en la app"})

Con `stream` vemos **qué nodo se ejecuta y qué devuelve** en cada paso: cada evento es un diccionario `{nodo: actualización}`.

In [ ]:
for part in triaje.stream({"texto": "Fallo en el pago: no se puede comprar"}, stream_mode="updates"):
    print(part)

Cada evento trae solo lo que devuelve ese nodo. El estado final es la suma de todos: fíjate en `pasos`,
donde el reducer ha ido concatenando lo de cada nodo.

In [ ]:
triaje.invoke({"texto": "Fallo en el pago: no se puede comprar"})

### ✍️ Ejercicio 1

Añade un nodo `resumir` al final del grafo que escriba en `pasos` un resumen como
`"resumen: SEV1 · Pagos · 3 pasos"`. Conéctalo entre `crear_ticket` y `END`.

Pista: copia la celda de construcción del grafo y cambia solo las aristas del final.

In [ ]:
# TODO: define `resumir` y vuelve a construir el grafo

<details><summary><b>Ver solución</b></summary>

```python
def resumir(state: Incidencia) -> dict:
    return {"pasos": [f"resumen: {state['severidad']} · {state['equipo']} · {len(state['pasos'])} pasos"]}

builder = StateGraph(Incidencia)
builder.add_node("clasificar", clasificar)
builder.add_node("avisar_guardia", avisar_guardia)
builder.add_node("crear_ticket", crear_ticket)
builder.add_node("resumir", resumir)
builder.add_edge(START, "clasificar")
builder.add_conditional_edges("clasificar", enrutar)
builder.add_edge("avisar_guardia", "crear_ticket")
builder.add_edge("crear_ticket", "resumir")
builder.add_edge("resumir", END)
triaje = builder.compile()
triaje.invoke({"texto": "La web está caída"})
```
</details>

## 1.5 · Ciclos: lo que una cadena (pipeline) no puede hacer

Los grafos de LangGraph pueden tener **ciclos**. Es justo lo que necesita un agente:
*pensar → actuar → observar → volver a pensar*. Aquí, un reintento con límite.

In [ ]:
class Reintento(TypedDict):
    intentos: int
    ok: bool


def llamar_pasarela(state: Reintento) -> dict:
    intentos = state["intentos"] + 1
    return {"intentos": intentos, "ok": intentos >= 3}  # simulamos que funciona al tercer intento


def decidir(state: Reintento) -> Literal["llamar_pasarela", "__end__"]:
    return END if state["ok"] or state["intentos"] >= 5 else "llamar_pasarela"


builder = StateGraph(Reintento)
builder.add_node("llamar_pasarela", llamar_pasarela)
builder.add_edge(START, "llamar_pasarela")
builder.add_conditional_edges("llamar_pasarela", decidir)
reintentos = builder.compile()
mostrar_grafo(reintentos)
reintentos.invoke({"intentos": 0, "ok": False})

> **Para recordar:** un nodo devuelve solo lo que cambia; los reducers deciden cómo se combina;
> las aristas condicionales y los ciclos dan el **control del flujo**. Eso es todo LangGraph:
> el resto (LLM, herramientas, memoria) son nodos y estado.

### 🚀 Reto opcional: `Command`

Un nodo también puede **actualizar el estado y decidir el siguiente paso a la vez** devolviendo
`Command(update=..., goto=...)`. Así no hace falta una función de enrutado aparte.

```python
from langgraph.types import Command

def clasificar(state: Incidencia) -> Command[Literal["avisar_guardia", "crear_ticket"]]:
    ...
    return Command(update={...}, goto="avisar_guardia" if severidad == "SEV1" else "crear_ticket")
```

Reescribe el grafo de triaje con `Command` y sin `add_conditional_edges`.